<a href="https://colab.research.google.com/github/yeye080/customs-hs-classifier/blob/main/clasificador_aduanero_inteligente.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# CLASIFICADOR INTELIGENTE DE PARTIDAS ARANCELARIAS (ENTORNO DE PRUEBAS)
# =====================================================================

import pandas as pd
from difflib import SequenceMatcher


# 1. BASE DE DATOS SINTÉTICA (Protección Datos y Privacidad)

# Uso de datos genéricos y simulados para evitar el uso de información confidencial.
datos_sinteticos = {
    'HS_Code': ['8471.30.00', '6403.91.11', '0901.21.00', '2204.21.11', '8517.13.00', '9503.00.49', '3926.90.97', '9018.90.84'],
    'Categoria_Generica': ['Equipos Informáticos', 'Calzado Industrial/Seguridad', 'Materia Prima Agrícola (Café)', 'Bebidas Alcohólicas Embotelladas', 'Dispositivos de Comunicación', 'Artículos de Entretenimiento (Plástico)', 'Manufacturas de Plástico Genéricas', 'Instrumental Médico de Diagnóstico'],
    'Keywords': ['ordenador laptop notebook pc computadora pantalla', 'zapatos botas calzado cuero montaña proteccion', 'cafe grano molido tostado cafeina', 'vino alcohol botella tinto blanco bodega', 'movil telefono smartphone celular wifi', 'juguete plastico figura muñeco infantil', 'caja plastico envase tubo pieza pvc', 'medico hospital pinza bisturi jeringa estetoscopio'],
    'Arancel_Estimado_%': [0.0, 8.0, 7.5, 12.0, 0.0, 4.7, 6.5, 0.0],
    'IVA_Estimado_%': [21.0, 21.0, 10.0, 21.0, 21.0, 21.0, 21.0, 4.0],
    'Restricciones_Simuladas': ['Certificado de Seguridad Electrónica', 'Inspección de Calidad de Materiales', 'Certificado Fitosanitario de Importación', 'Licencia de Importación de Alcoholes', 'Homologación de Frecuencias', 'Certificación Infantil de No Toxicidad', 'Declaración de Impacto Ambiental (Plásticos)', 'Licencia de Importación Sanitaria / Marcado Médico']
}

df_aduanas_mock = pd.DataFrame(datos_sinteticos)



# 2. MOTOR DE BÚSQUEDA APROXIMADA (Fuzzy Matching)

def calcular_similitud(texto1, texto2):
    """Calcula el ratio de parecido entre dos cadenas de texto."""
    return SequenceMatcher(None, texto1.lower(), texto2.lower()).ratio()

def clasificador_aduanero_inteligente(busqueda_usuario, valor_mercancia=0.0, coste_envio=0.0):
    palabra_buscar = busqueda_usuario.lower().strip()
    mejor_coincidencia = None
    mejor_puntuacion = 0.0

    # Buscar la mejor coincidencia comparando con categorías y palabras clave
    for _, fila in df_aduanas_mock.iterrows():
        score_categoria = calcular_similitud(palabra_buscar, fila['Categoria_Generica'])

        score_keywords = 0.0
        for kw in fila['Keywords'].split():
            score_kw = calcular_similitud(palabra_buscar, kw)
            if score_kw > score_keywords:
                score_keywords = score_kw

        score_final = max(score_categoria, score_keywords)

        if score_final > mejor_puntuacion:
            mejor_puntuacion = score_final
            mejor_coincidencia = fila

    # Umbral de confianza (mínimo 60% de coincidencia)
    if mejor_coincidencia is None or mejor_puntuacion < 0.6:
        print(f"❌ No se pudo clasificar '{busqueda_usuario}' con suficiente seguridad.")
        print("💡 Prueba con términos como: 'computadora', 'bota', 'café', 'celular', 'bisturí' o 'botella'.")
        return


    # 3. LIQUIDACIÓN DE IMPUESTOS EN ADUANAS (CIF)

    valor_cif = valor_mercancia + coste_envio
    arancel_pagar = valor_cif * (mejor_coincidencia['Arancel_Estimado_%'] / 100.0)
    base_iva = valor_cif + arancel_pagar
    iva_pagar = base_iva * (mejor_coincidencia['IVA_Estimado_%'] / 100.0)
    total_impuestos = arancel_pagar + iva_pagar


    # 4. SALIDA DE DATOS FORMATEADA

    print("=" * 70)
    print("🛡️  CLASIFICACIÓN ARANCELARIA AUTOMÁTICA")
    print("=" * 70)
    print(f"🔍 Búsqueda:                 '{busqueda_usuario}'")
    print(f"🎯 Categoría Identificada:   {mejor_coincidencia['Categoria_Generica']} ({mejor_puntuacion*100:.1f}% confianza)")
    print(f"🔢 Código HS Simulado:       {mejor_coincidencia['HS_Code']}")
    print(f"📋 Requisitos de Aduana:     {mejor_coincidencia['Restricciones_Simuladas']}")
    print("-" * 70)

    if valor_mercancia > 0:
        print("💰 SIMULACIÓN DE LIQUIDACIÓN TRIBUTARIA (CIF):")
        print(f"• Valor de Mercancía:        {valor_mercancia:.2f} €")
        print(f"• Logística (Flete/Seguro):   {coste_envio:.2f} €")
        print(f"• Arancel Estimado ({mejor_coincidencia['Arancel_Estimado_%']}%):  {arancel_pagar:.2f} €")
        print(f"• IVA Estimado ({mejor_coincidencia['IVA_Estimado_%']}%):      {iva_pagar:.2f} €")
        print(f"🚨 TOTAL LIQUIDACIÓN ADUANA: {total_impuestos:.2f} €")
        print("=" * 70)


# 5. PRUEBAS DE FUNCIONAMIENTO EN VIVO

# Prueba 1: Buscando con error de escritura ("compuadora")
clasificador_aduanero_inteligente("compuadora", valor_mercancia=3500.0, coste_envio=250.0)

print("\n")

# Prueba 2: Buscando por un sinónimo que no está en el título principal ("botas")
clasificador_aduanero_inteligente("botas", valor_mercancia=12000.0, coste_envio=1100.0)

🛡️  CLASIFICACIÓN ARANCELARIA AUTOMÁTICA
🔍 Búsqueda:                 'compuadora'
🎯 Categoría Identificada:   Equipos Informáticos (95.2% confianza)
🔢 Código HS Simulado:       8471.30.00
📋 Requisitos de Aduana:     Certificado de Seguridad Electrónica
----------------------------------------------------------------------
💰 SIMULACIÓN DE LIQUIDACIÓN TRIBUTARIA (CIF):
• Valor de Mercancía:        3500.00 €
• Logística (Flete/Seguro):   250.00 €
• Arancel Estimado (0.0%):  0.00 €
• IVA Estimado (21.0%):      787.50 €
🚨 TOTAL LIQUIDACIÓN ADUANA: 787.50 €


🛡️  CLASIFICACIÓN ARANCELARIA AUTOMÁTICA
🔍 Búsqueda:                 'botas'
🎯 Categoría Identificada:   Calzado Industrial/Seguridad (100.0% confianza)
🔢 Código HS Simulado:       6403.91.11
📋 Requisitos de Aduana:     Inspección de Calidad de Materiales
----------------------------------------------------------------------
💰 SIMULACIÓN DE LIQUIDACIÓN TRIBUTARIA (CIF):
• Valor de Mercancía:        12000.00 €
• Logística (Flete/Seguro):

In [ ]:
## Interfaz Interactiva y Visualización de Costes (v1.1)

In [4]:

# CLASIFICADOR ADUANERO INTELIGENTE V1.1 (INTERFAZ Y GRÁFICOS)

import pandas as pd
from difflib import SequenceMatcher
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

# 1. Base de datos sintética
datos_sinteticos = {
    'HS_Code': ['8471.30.00', '6403.91.11', '0901.21.00', '2204.21.11', '8517.13.00', '9503.00.49', '3926.90.97', '9018.90.84'],
    'Categoria_Generica': ['Equipos Informáticos', 'Calzado Industrial/Seguridad', 'Materia Prima Agrícola (Café)', 'Bebidas Alcohólicas Embotelladas', 'Dispositivos de Comunicación', 'Artículos de Entretenimiento (Plástico)', 'Manufacturas de Plástico Genéricas', 'Instrumental Médico de Diagnóstico'],
    'Keywords': ['ordenador laptop notebook pc computadora pantalla', 'zapatos botas calzado cuero montaña proteccion', 'cafe grano molido tostado cafeina', 'vino alcohol botella tinto blanco bodega', 'movil telefono smartphone celular wifi', 'juguete plastico figura muñeco infantil', 'caja plastico envase tubo pieza pvc', 'medico hospital pinza bisturi jeringa estetoscopio'],
    'Arancel_Estimado_%': [0.0, 8.0, 7.5, 12.0, 0.0, 4.7, 6.5, 0.0],
    'IVA_Estimado_%': [21.0, 21.0, 10.0, 21.0, 21.0, 21.0, 21.0, 4.0],
    'Restricciones_Simuladas': ['Certificado de Seguridad Electrónica', 'Inspección de Calidad de Materiales', 'Certificado Fitosanitario de Importación', 'Licencia de Importación de Alcoholes', 'Homologación de Frecuencias', 'Certificación Infantil de No Toxicidad', 'Declaración de Impacto Ambiental (Plásticos)', 'Licencia de Importación Sanitaria / Marcado Médico']
}
df_aduanas_mock = pd.DataFrame(datos_sinteticos)

# 2. Cálculo matemático y generación del gráfico
def procesar_aduana(busqueda_usuario, valor_mercancia, coste_envio):
    palabra_buscar = busqueda_usuario.lower().strip()
    mejor_coincidencia = None
    mejor_puntuacion = 0.0

    # Algoritmo Fuzzy Matching
    for _, fila in df_aduanas_mock.iterrows():
        score_categoria = SequenceMatcher(None, palabra_buscar, fila['Categoria_Generica'].lower()).ratio()
        score_keywords = 0.0
        for kw in fila['Keywords'].split():
            score_kw = SequenceMatcher(None, palabra_buscar, kw).ratio()
            if score_kw > score_keywords:
                score_keywords = score_kw
        score_final = max(score_categoria, score_keywords)

        if score_final > mejor_puntuacion:
            mejor_puntuacion = score_final
            mejor_coincidencia = fila

    if mejor_coincidencia is None or mejor_puntuacion < 0.6:
        print(f"❌ No se pudo clasificar '{busqueda_usuario}' con suficiente seguridad.")
        print("💡 Sugerencias de prueba: 'computadora', 'bota', 'café', 'celular', 'bisturí' o 'botella'.")
        return

    # Cálculos reales de liquidación de aduanas
    valor_cif = valor_mercancia + coste_envio
    arancel_pagar = valor_cif * (mejor_coincidencia['Arancel_Estimado_%'] / 100.0)
    base_iva = valor_cif + arancel_pagar
    iva_pagar = base_iva * (mejor_coincidencia['IVA_Estimado_%'] / 100.0)
    total_impuestos = arancel_pagar + iva_pagar
    total_operacion = valor_cif + total_impuestos

    # Mostrar resultados formato texto
    print("=" * 70)
    print("🛡️  RESULTADO DE LA CLASIFICACIÓN ARANCELARIA AUTOMÁTICA")
    print("=" * 70)
    print(f"🔍 Búsqueda del usuario:     '{busqueda_usuario}'")
    print(f"🎯 Categoría Identificada:   {mejor_coincidencia['Categoria_Generica']} ({mejor_puntuacion*100:.1f}% confianza)")
    print(f"🔢 Código HS Simulado:       {mejor_coincidencia['HS_Code']}")
    print(f"📋 Requisitos de Importación: {mejor_coincidencia['Restricciones_Simuladas']}")
    print("-" * 70)
    print(f"💰 LIQUIDACIÓN TRIBUTARIA Y LOGÍSTICA:")
    print(f"• Coste Mercancía (FOB):     {valor_mercancia:,.2f} €")
    print(f"• Coste Logístico (Flete):   {coste_envio:,.2f} €")
    print(f"• Arancel a Pagar ({mejor_coincidencia['Arancel_Estimado_%']}%):  {arancel_pagar:,.2f} €")
    print(f"• IVA a Pagar ({mejor_coincidencia['IVA_Estimado_%']}%):      {iva_pagar:,.2f} €")
    print(f"🚨 TOTAL LIQUIDACIÓN ADUANA: {total_impuestos:,.2f} €")
    print(f"💵 COSTE TOTAL DE OPERACIÓN:  {total_operacion:,.2f} €")
    print("=" * 70)

    # Generación de la visualización gráfica
    conceptos = ['Mercancía (FOB)', 'Transporte/Seguro', 'Arancel aduanero', 'IVA de Importación']
    costes = [valor_mercancia, coste_envio, arancel_pagar, iva_pagar]

    # Filtro valores en cero para que el gráfico no se solape
    conceptos_filtrados = [c for c, v in zip(conceptos, costes) if v > 0]
    costes_filtrados = [v for v in costes if v > 0]

    # Paleta de colores profesionales
    colores = ['#2b5c8f', '#4682b4', '#e06666', '#f4cccc']

    plt.figure(figsize=(8, 5))
    plt.pie(
        costes_filtrados,
        labels=conceptos_filtrados,
        autopct='%1.1f%%',
        startangle=140,
        colors=colores[:len(costes_filtrados)],
        wedgeprops={'edgecolor': 'white', 'linewidth': 2}
    )
    plt.title('Distribución Total de Costes de la Importación', fontsize=14, fontweight='bold', pad=20)
    plt.tight_layout()
    plt.show()

# 3. Construcción del Formulario Interactivo (UI Widgets)
salida = widgets.Output()

input_producto = widgets.Text(value='bota de seguridad', description='Producto:', style={'description_width': 'initial'})
input_valor = widgets.FloatText(value=5000.0, description='Valor FOB (€):', style={'description_width': 'initial'})
input_flete = widgets.FloatText(value=600.0, description='Flete/Seguro (€):', style={'description_width': 'initial'})
boton_calcular = widgets.Button(description='Calcular Despacho', button_style='primary', icon='calculator')

def ejecutar_calculo(b):
    with salida:
        clear_output(wait=True)
        procesar_aduana(input_producto.value, input_valor.value, input_flete.value)

boton_calcular.on_click(ejecutar_calculo)

# Renderizado de los elementos visuales en Colab
print("🛠️ INTERFAZ DE SIMULACIÓN DE ADUANAS LISTA")
print("Modifica los valores del formulario y haz clic en 'Calcular Despacho':")
display(widgets.VBox([input_producto, input_valor, input_flete, boton_calcular]), salida)


🛠️ INTERFAZ DE SIMULACIÓN DE ADUANAS LISTA
Modifica los valores del formulario y haz clic en 'Calcular Despacho':


Output()

In [ ]:
## Simulador de Logística y Aduanas PRO (v2.0)

En esta versión definitiva, escalamos el clasificador arancelario básico para convertirlo en un simulador de operaciones internacionales real. Hemos añadido tres características clave:

1. **Conversor de Divisas en Tiempo Real:** Integración con una API de tipos de cambio para calcular aranceles en Euros (EUR) aunque la mercancía se compre en Dólares (USD), Libras (GBP) o Yuanes (CNY).
2. **Semáforo de Riesgo Aduanero:** Alertas visuales basadas en el tipo de producto (Canal Verde, inspecciones fitosanitarias, impuestos especiales o control de juguetes).
3. **Cálculo de Cubicaje (Peso Tasable):** Implementación de las fórmulas estándar de la IATA para calcular el peso volumétrico y determinar si la naviera o aerolínea facturará por volumen o por peso real.

In [7]:
# SIMULADOR ADUANERO E INCOTERMS PRO (V2.0)


import pandas as pd
from difflib import SequenceMatcher
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
import urllib.request
import json

# 1. BASE DE DATOS SINTÉTICA CON SEMÁFORO DE RIESGO ADUANERO
datos_sinteticos = {
    'HS_Code': ['8471.30.00', '6403.91.11', '0901.21.00', '2204.21.11', '8517.13.00', '9503.00.49', '3926.90.97', '9018.90.84'],
    'Categoria_Generica': ['Equipos Informáticos', 'Calzado Industrial/Seguridad', 'Materia Prima Agrícola (Café)', 'Bebidas Alcohólicas Embotelladas', 'Dispositivos de Comunicación', 'Artículos de Entretenimiento (Plástico)', 'Manufacturas de Plástico Genéricas', 'Instrumental Médico de Diagnóstico'],
    'Keywords': ['ordenador laptop notebook pc computadora pantalla', 'zapatos botas calzado cuero montaña proteccion', 'cafe grano molido tostado cafeina', 'vino alcohol botella tinto blanco bodega', 'movil telefono smartphone celular wifi', 'juguete plastico figura muñeco infantil', 'caja plastico envase tubo pieza pvc', 'medico hospital pinza bisturi jeringa estetoscopio'],
    'Arancel_Estimado_%': [0.0, 8.0, 7.5, 12.0, 0.0, 4.7, 6.5, 0.0],
    'IVA_Estimado_%': [21.0, 21.0, 10.0, 21.0, 21.0, 21.0, 21.0, 4.0],
    'Restricciones_Simuladas': ['Certificado de Seguridad Electrónica', 'Inspección de Calidad de Materiales', 'Certificado Fitosanitario de Importación', 'Licencia de Importación de Alcoholes', 'Homologación de Frecuencias', 'Certificación Infantil de No Toxicidad', 'Declaración de Impacto Ambiental (Plásticos)', 'Licencia de Importación Sanitaria / Marcado Médico'],
    'Riesgo_Nivel': ['MEDIO', 'BAJO', 'ALTO', 'ALTO', 'MEDIO', 'ALTO', 'BAJO', 'ALTO'],
    'Riesgo_Mensaje': [
        '⚠️ Requiere Marcado CE. Inspecciones documentales aleatorias en aduana.',
        '🟢 Canal Verde habitual. Trámite rápido si el certificado de origen es correcto.',
        '🚨 CONTROL FITOSANITARIO Obligatorio. Riesgo de retención física de la mercancía si falta el certificado de sanidad en origen.',
        '🚨 IMPUESTOS ESPECIALES. Gestión de precintas fiscales obligatoria. Alta probabilidad de inspección física.',
        '⚠️ Homologación de telecomunicaciones requerida para evitar bloqueos por seguridad.',
        '🚨 CONTROL DE JUGUETES (Sanidad). Ensayos de toxicidad estrictos en frontera. Alto riesgo de bloqueo sin tests de laboratorio.',
        '🟢 Canal Verde habitual. Bajo riesgo de retención si la declaración de plásticos está en regla.',
        '🚨 LICENCIA SANITARIA. Inspección obligatoria de Farmacia en Aduana. Solo importadores autorizados por sanidad.'
    ]
}
df_aduanas_mock = pd.DataFrame(datos_sinteticos)

# 2. CONVERSOR DE DIVISAS EN TIEMPO REAL
def obtener_tipo_cambio(moneda_origen, moneda_destino="EUR"):
    if moneda_origen == moneda_destino:
        return 1.0
    try:
        url = f"https://open.er-api.com/v6/latest/{moneda_origen}"
        response = urllib.request.urlopen(url)
        data = json.loads(response.read().decode())
        return data["rates"].get(moneda_destino, 1.0)
    except Exception:
        # Fallback de seguridad
        tasas_estaticas = {"USD": 0.92, "GBP": 1.18, "CNY": 0.13}
        return tasas_estaticas.get(moneda_origen, 1.0)

# 3. CÁLCULO DE CUBICAJE
def calcular_peso_tasable(largo, ancho, alto, peso_real, tipo_transporte):
    volumen_m3 = (largo * ancho * alto) / 1000000.0
    if tipo_transporte == "Aéreo":
        peso_volumetrico = (largo * ancho * alto) / 6000.0
    else:
        peso_volumetrico = (largo * ancho * alto) / 3000.0
    peso_tasable = max(peso_real, peso_volumetrico)
    return peso_volumetrico, peso_tasable, volumen_m3

# 4. MOTOR LÓGICO DE INCOTERMS E IMPUESTOS
def procesar_aduana_incoterms(busqueda, valor_factura, moneda, incoterm, transporte_origen, aduana_exportacion, flete_internacional, gastos_llegada, largo, ancho, alto, peso_real, tipo_transporte):
    palabra_buscar = busqueda.lower().strip()
    mejor_coincidencia = None
    mejor_puntuacion = 0.0

    # Algoritmo Fuzzy Matching
    for _, fila in df_aduanas_mock.iterrows():
        score_categoria = SequenceMatcher(None, palabra_buscar, fila['Categoria_Generica'].lower()).ratio()
        score_keywords = 0.0
        for kw in fila['Keywords'].split():
            score_kw = SequenceMatcher(None, palabra_buscar, kw).ratio()
            if score_kw > score_keywords:
                score_keywords = score_kw
        score_final = max(score_categoria, score_keywords)

        if score_final > mejor_puntuacion:
            mejor_puntuacion = score_final
            mejor_coincidencia = fila

    if mejor_coincidencia is None or mejor_puntuacion < 0.6:
        print(f"❌ No se pudo clasificar '{busqueda}' con suficiente seguridad.")
        print("💡 Sugerencias de prueba: 'computadora', 'bota', 'café', 'celular', 'bisturí' o 'botella'.")
        return

    # Obtención de tasas de cambio a Euros
    tasa = obtener_tipo_cambio(moneda, "EUR")

    # Conversión de todos los importes introducidos a EUR
    val_factura_eur = valor_factura * tasa
    t_origen_eur = transporte_origen * tasa
    a_exp_eur = aduana_exportacion * tasa
    flete_eur = flete_internacional * tasa
    llegada_eur = gastos_llegada * tasa

    # Cálculo de Cubicaje
    p_vol, p_tasable, vol_m3 = calcular_peso_tasable(largo, ancho, alto, peso_real, tipo_transporte)

    # REGLAS DE CONSTRUCCIÓN DEL VALOR DE ADUANA (VALOR CIF / ADUANA) SEGÚN EL INCOTERM
    # El objetivo es calcular el valor de la mercancía puesta sobre el puerto de destino (CIF).
    if incoterm == "EXW":
        # EXW: Factura no incluye nada. Sumamos todo para llegar a CIF.
        valor_cif = val_factura_eur + t_origen_eur + a_exp_eur + flete_eur
        paga_comprador_logistica = t_origen_eur + a_exp_eur + flete_eur + llegada_eur
    elif incoterm in ["FCA", "FOB", "FAS"]:
        # FOB/FCA: Factura incluye transporte en origen y aduana de exportación. Sumamos flete.
        valor_cif = val_factura_eur + flete_eur
        paga_comprador_logistica = flete_eur + llegada_eur
    elif incoterm in ["CFR", "CPT", "CIF", "CIP"]:
        # CIF/CIP/CPT: Factura ya incluye el flete internacional y el seguro. No se suma flete.
        valor_cif = val_factura_eur
        paga_comprador_logistica = llegada_eur
    elif incoterm in ["DAP", "DPU"]:
        # DAP: Se entrega en destino, pero el comprador despacha aduana. El valor de factura ya cubre flete.
        valor_cif = val_factura_eur
        paga_comprador_logistica = 0.0
    elif incoterm == "DDP":
        # DDP: El vendedor ya paga todo (aranceles incluidos). El valor CIF es el de factura restando impuestos estimados.
        # Hacemos estimación hacia atrás para deducir el CIF neto teórico
        porcentaje_impuestos_total = (mejor_coincidencia['Arancel_Estimado_%'] + mejor_coincidencia['IVA_Estimado_%']) / 100.0
        valor_cif = val_factura_eur / (1.0 + porcentaje_impuestos_total)
        paga_comprador_logistica = 0.0

    # Cálculos de Liquidación de Impuestos
    arancel_pagar = valor_cif * (mejor_coincidencia['Arancel_Estimado_%'] / 100.0)
    base_iva = valor_cif + arancel_pagar
    iva_pagar = base_iva * (mejor_coincidencia['IVA_Estimado_%'] / 100.0)
    total_impuestos = arancel_pagar + iva_pagar

    # Quién asume el coste de los impuestos de aduana
    impuestos_comprador = 0.0 if incoterm == "DDP" else total_impuestos

    # Coste final total que sale del bolsillo de nuestra empresa (Mercancía + Logística asumida + Impuestos asumidos)
    total_coste_empresa = val_factura_eur + paga_comprador_logistica + impuestos_comprador

    # --- SALIDA FORMATEADA ---
    print("=" * 80)
    print(f"🛡️  SIMULADOR LOGÍSTICO Y ADUANERO INCOTERMS PRO")
    print("=" * 80)
    print(f"🔍 Producto:               '{busqueda}'")
    print(f"🎯 Categoría Identificada:  {mejor_coincidencia['Categoria_Generica']} ({mejor_puntuacion*100:.1f}% confianza)")
    print(f"📦 Regla Incoterm Compra:  [{incoterm}]")
    print(f"🔢 Código HS Simulado:      {mejor_coincidencia['HS_Code']}")
    print("-" * 80)

    print(f"🚦 SEMÁFORO DE RIESGO ADUANERO: [{mejor_coincidencia['Riesgo_Nivel']}]")
    print(f"   {mejor_coincidencia['Riesgo_Mensaje']}")
    print("-" * 80)

    print(f"📦 DATOS FÍSICOS Y CUBICAJE (Transporte: {tipo_transporte})")
    print(f"   • Volumen de carga:     {vol_m3:.3f} m³")
    print(f"   • Peso Real (Báscula):  {peso_real:,.1f} kg")
    print(f"   • Peso Volumétrico:     {p_vol:,.1f} kg")
    print(f"   👉 PESO TASABLE COBRADO: {p_tasable:,.1f} kg (Tarifa por {'Volumen' if p_vol > peso_real else 'Peso'})")
    print("-" * 80)

    print(f"💰 LIQUIDACIÓN DE IMPUESTOS EN ADUANA (EUR)")
    if moneda != "EUR":
        print(f"   * Tipo de cambio: 1 {moneda} = {tasa:.4f} EUR")
    print(f"   • Valor de Factura:     {val_factura_eur:,.2f} €  ({valor_factura:,.2f} {moneda})")
    print(f"   👉 BASE DE ADUANA (CIF): {valor_cif:,.2f} €  (Valor calculado para aplicar aranceles)")
    print(f"   • Arancel a Pagar ({mejor_coincidencia['Arancel_Estimado_%']}%):  {arancel_pagar:,.2f} €")
    print(f"   • IVA de Importación ({mejor_coincidencia['IVA_Estimado_%']}%):  {iva_pagar:,.2f} €")
    print(f"   🚨 IMPUESTOS ADUANEROS TOTALES: {total_impuestos:,.2f} €")
    print("-" * 80)

    print(f"📊 DESGLOSE DE COSTES TOTALES PARA NUESTRA EMPRESA (EUR)")
    print(f"   • Compra de la Mercancía (Factura):  {val_factura_eur:,.2f} €")
    print(f"   • Logística Extra Pagada por nosotros: {paga_comprador_logistica:,.2f} €")
    print(f"   • Impuestos de Importación Pagados:    {impuestos_comprador:,.2f} €")
    print(f"   💵 COSTE FINAL PUESTO EN ALMACÉN:      {total_coste_empresa:,.2f} €")
    print("=" * 80)

    # Gráfica de distribución de costes
    conceptos = ['Factura Proveedor', 'Transporte/Gastos Extra', 'Impuestos Aduana']
    costes = [val_factura_eur, paga_comprador_logistica, impuestos_comprador]

    plt.figure(figsize=(8, 4))
    plt.pie(
        [v for v in costes if v > 0],
        labels=[c for c, v in zip(conceptos, costes) if v > 0],
        autopct='%1.1f%%',
        startangle=140,
        colors=['#2b5c8f', '#ff9999', '#66b3ff'],
        wedgeprops={'edgecolor': 'white', 'linewidth': 2}
    )
    plt.title(f'Distribución del Coste Total de Adquisición - Incoterm {incoterm}', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()

# 5. DISEÑO DE LA INTERFAZ INTERACTIVA (IPYWIDGETS)
salida_incoterms = widgets.Output()

html_titulo1 = widgets.HTML("<h4>📝 1. Compra y Producto</h4>")
in_prod = widgets.Text(value='bota de seguridad', description='Producto:')
in_factura = widgets.FloatText(value=10000.0, description='Factura:')
in_moneda = widgets.Dropdown(options=['EUR', 'USD', 'GBP', 'CNY'], value='USD', description='Divisa:')
in_incoterm = widgets.Dropdown(options=['EXW', 'FOB', 'CIF', 'DDP'], value='EXW', description='Incoterm:')

html_titulo2 = widgets.HTML("<h4>🚛 2. Desglose de Gastos Logísticos (Poner el total estimado)</h4>")
in_trans_origen = widgets.FloatText(value=300.0, description='Camión Origen:')
in_exp = widgets.FloatText(value=150.0, description='Aduana Export:')
in_flete = widgets.FloatText(value=1500.0, description='Flete + Seguro:')
in_llegada = widgets.FloatText(value=200.0, description='Gatos Destino:')

html_titulo3 = widgets.HTML("<h4>📦 3. Dimensiones de la Carga (Cubicaje)</h4>")
in_largo = widgets.FloatText(value=120.0, description='Largo (cm):')
in_ancho = widgets.FloatText(value=80.0, description='Ancho (cm):')
in_alto = widgets.FloatText(value=160.0, description='Alto (cm):')
in_peso = widgets.FloatText(value=150.0, description='Peso real (kg):')
in_tipo_transp = widgets.Dropdown(options=['Aéreo', 'Marítimo/Terrestre'], value='Marítimo/Terrestre', description='Modo:')

boton_calcular = widgets.Button(description='Simular Operación Incoterm', button_style='danger', icon='anchor')

def ejecutar_simulacion_incoterm(b):
    with salida_incoterms:
        clear_output(wait=True)
        procesar_aduana_incoterms(
            in_prod.value, in_factura.value, in_moneda.value, in_incoterm.value,
            in_trans_origen.value, in_exp.value, in_flete.value, in_llegada.value,
            in_largo.value, in_ancho.value, in_alto.value, in_peso.value, in_tipo_transp.value
        )

boton_calcular.on_click(ejecutar_simulacion_incoterm)

# Estructurar la visualización del formulario en pestañas o columnas
formulario_pro = widgets.VBox([
    html_titulo1, widgets.HBox([in_prod, in_incoterm]), widgets.HBox([in_factura, in_moneda]),
    html_titulo2, widgets.HBox([in_trans_origen, in_exp]), widgets.HBox([in_flete, in_llegada]),
    html_titulo3, widgets.HBox([in_largo, in_ancho]), widgets.HBox([in_alto, in_peso, in_tipo_transp]),
    widgets.Label(''), boton_calcular
])

print("⚓ SISTEMA LOGÍSTICO COMPLETO CON REGLAS INCOTERMS INICIADO")
display (formulario_pro, salida_incoterms)

⚓ SISTEMA LOGÍSTICO COMPLETO CON REGLAS INCOTERMS INICIADO


Output()